## In-Class Lecture 19-2 - Registers vs. Static vs. Dynamic Memory Timing

### Running the initial Dynamic Array Comparison

First, run the following command:

    make dyn_array

You will get an output similar to the following:

    $ make dyn_array
    gcc -std=c2x -g -Wall -Wextra -Wconversion -Wshadow -Werror dyn_array.c -o dyn_array
    ./dyn_array
    Total Clock Pulses: 2752795
    Avg. execution time: 0.002753 sec
    ----------------------------------------------

Notice that the Production Quality Compilation flag <code>-O2</code> have been removed. We will discuss later how this flag impacts the performance of the code

We will investigate <code>dyn_array.c</code>. We see that this code performs 1000 tests where a dynamic array of length 1,000,000 is allocated, and the loop for 1,000,000 Fibonacci numbers are calculated
<ul><li>Note: The 1,000,000th is much too large for any of the types we have available in C. This example is presented as an example for timing demonstrations only. </li></ul>

### Improving the performance with the <code>register</code> keyword

If we want to explicitly use fast registers, we will eliminate as many accesses to the dynamic memory as possible. We will only access the memory when we absolutely have to. 

    void fibonacci()
    {

        // Allocate the memory once
        unsigned int* results_buffer = (unsigned int *)malloc( NMAX * sizeof(unsigned int) );

        register unsigned int first_num = 1;

        register unsigned int second_num = 1;

        register unsigned int third_num; // No need to re-allocate ever loop

        register long unsigned int iter;
        for (iter = 2; iter < NMAX; iter++) {

            third_num = first_num + second_num;

            results_buffer[ iter ] = third_num;

            first_num = second_num;

            second_num = third_num;
        }

        free(results_buffer);

    }

Notice the significant improvement when we compile and run. For this specific run, we have improved from 0.002753 to 0.000715

    $ make dyn_array_reg
    gcc -std=c2x -g -Wall -Wextra -Wconversion -Wshadow -Werror dyn_array_reg.c -o dyn_array_reg
    ./dyn_array_reg
    Total Clock Pulses: 715456
    Avg. execution time: 0.000715 sec
    ----------------------------------------------


### Comparing with static memory

For static memory, we will not allocate or free memory. The static array is allocated at compile time, so the time is slightly less.

    void fibonacci()
    {

        unsigned int results_buffer[NMAX];

        /* First array element  */
        results_buffer[0] = 1;

        /* Second array element */
        results_buffer[1] = 1;

        long unsigned int iter;
        for (iter = 2; iter < NMAX; iter++) {

            results_buffer[ iter ] = results_buffer[ iter-2 ] + results_buffer[ iter - 1 ];

        }

    }

And here is the run time

    $ make stat_array
    gcc -std=c2x -g -Wall -Wextra -Wconversion -Wshadow -Werror stat_array.c -o stat_array
    ./stat_array
    Total Clock Pulses: 2296794
    Avg. execution time: 0.002297 sec
    ----------------------------------------------

### Updating with registers

Finally, we will update with registers

